## Вычисляем  регрессию для признака IC50 используя  разные наборы  данных из исходного датасет, а так же пробуем  прогнать  исходный очищенный датасет  без понижения размерности

для качества  оценки моделей буду использовать в основном  R^2 (coefficient of determination) — доля объяснённой дисперсии, но включим в набор оценки и RMSE

In [55]:

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import AdaBoostRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from sklearn.feature_selection import SelectKBest, f_classif
from skfeature.function.similarity_based import fisher_score

from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import RFE

from sklearn.preprocessing import StandardScaler, QuantileTransformer

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.feature_selection import RFECV, SelectFromModel, mutual_info_regression
from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
import seaborn as sns
from scipy import stats
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LinearRegression

In [48]:
#Загружаем предобработанный датасет и  данные отобранные на  шаге исследования (EDA) различными методами
drugs_all = pd.read_csv('drugs_fin_clean.csv') #
select_sum = pd.read_csv('selected_sum.csv')
select_cP_tF = pd.read_csv('select_cP_tF.csv')
selected_features_MutRegr = pd.read_csv('selected_features_MutRegrv2.csv')
select_RF = pd.read_csv('select_RF.csv')
selected_features_Lasso = pd.read_csv('selected_features_Lasso.csv')
selected_features_Forward = pd.read_csv('selected_features_Forward.csv')
selected_features_Backwards = pd.read_csv('selected_features_Backwards.csv')

In [ ]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

#X = drugs_all[selected_features_Backwards['0']].copy()
#X['CC50, mM'] = drugs_all['CC50, mM']
X = drugs_all.drop(columns=['IC50, mM','SI']).copy()  # пробуем на полном датасете
y = drugs_all['IC50, mM'].values

# Разделение
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}\n")

#Масштабирование (сохраняем для разных моделей)

# Для линейных моделей и SVR нужна стандартизация
scaler_standard = StandardScaler()
X_train_std = scaler_standard.fit_transform(X_train)
X_test_std = scaler_standard.transform(X_test)

# Для деревьев масштабирование не нужно
X_train_raw = X_train.values
X_test_raw = X_test.values


#Базовое сравнение всех моделей

print("сравнение моделей \n")

models = {
    # Линейные модели
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(random_state=RANDOM_STATE),
    'Lasso': Lasso(random_state=RANDOM_STATE),
    
    # Деревья и ансамбли
    'Decision Tree': DecisionTreeRegressor(random_state=RANDOM_STATE),
    'Random Forest': RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(random_state=RANDOM_STATE),
    'AdaBoost': AdaBoostRegressor(random_state=RANDOM_STATE),
    
    # Современные бустинги (для хим. данных больше подходят - информация есть такая )
    'XGBoost': XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1, verbosity=0),
    'LightGBM': LGBMRegressor(random_state=RANDOM_STATE, n_jobs=-1, verbose=-1),
    'CatBoost': CatBoostRegressor(random_state=RANDOM_STATE, verbose=False),
    
    # Другие(для разнообразия)
    'KNN': KNeighborsRegressor(),
    'SVR': SVR()
}

# Кому какое масштабирование нужно
scaler_needed = ['Linear Regression', 'Ridge', 'Lasso', 
                 'KNN', 'SVR']

results = []

for name, model in models.items():
    # Выбираем данные
    if name in scaler_needed:
        X_tr, X_te = X_train_std, X_test_std
    else:
        X_tr, X_te = X_train_raw, X_test_raw
    
    try:
        # Кросс-валидация
        cv_scores = cross_val_score(model, X_tr, y_train, cv=5, scoring='r2', n_jobs=-1)
        
        # Обучение и тест
        model.fit(X_tr, y_train)
        y_pred = model.predict(X_te)
        
        r2 = r2_score(y_test, y_pred)
        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        
        results.append({
            'Model': name,
            'CV R^2 (mean)': cv_scores.mean(),
            'CV R^2 (std)': cv_scores.std(),
            'Test R^2': r2,
            'Test MAE': mae,
            'Test RMSE': rmse,
        })
        
        print(f"{name:20} | CV R^2 = {cv_scores.mean():.4f} ± {cv_scores.std():.4f} | Test R^2 = {r2:.4f}")
        
    except Exception as e:
        print(f"{name:20} | Ошибка: {str(e)[:50]}")
        continue

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('Test R^2', ascending=False)

print("\n=== Топ-5 моделей по Test R^2 ===")
print(results_df.head(5).to_string(index=False))

Train: (775, 211), Test: (194, 211)

сравнение моделей 

Linear Regression    | CV R^2 = -209486.3691 ± 418970.6140 | Test R^2 = 0.2263
Ridge                | CV R^2 = 0.2594 ± 0.1856 | Test R^2 = 0.4483
Lasso                | CV R^2 = 0.1429 ± 0.0305 | Test R^2 = 0.1458
Decision Tree        | CV R^2 = 0.1296 ± 0.1947 | Test R^2 = 0.3941
Random Forest        | CV R^2 = 0.5451 ± 0.0506 | Test R^2 = 0.6300
Gradient Boosting    | CV R^2 = 0.5448 ± 0.0589 | Test R^2 = 0.6607
AdaBoost             | CV R^2 = 0.5042 ± 0.0568 | Test R^2 = 0.6121
XGBoost              | CV R^2 = 0.4474 ± 0.0695 | Test R^2 = 0.5451
LightGBM             | CV R^2 = 0.5114 ± 0.0515 | Test R^2 = 0.6167
CatBoost             | CV R^2 = 0.5484 ± 0.0470 | Test R^2 = 0.6222
KNN                  | CV R^2 = 0.3348 ± 0.1117 | Test R^2 = 0.4251
SVR                  | CV R^2 = 0.4007 ± 0.0655 | Test R^2 = 0.5378

=== Топ-5 моделей по Test R^2 ===
            Model  CV R^2 (mean)  CV R^2 (std)  Test R^2  Test MAE  Test RMSE
Gra